# JRA-3Q 海面更正気圧（気圧配置）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

## セッション切れ対策について
- ダウンロード先を **Googleドライブ** にします。Colabのセッションが切れても、途中まで保存したファイルはドライブに残ります。
- ダウンロードスクリプトは **既にあるファイルはスキップ** する仕組みになっているので、セッションが切れて Colab に接続し直しても、上から順に「①→②→③→④」のセルを実行し直すだけで、続きから自動的に再開されます（最初からやり直しにはなりません）。
- 全185ヶ月・約15GBあるため、Colab無料枠のセッション制限（最大12時間・無操作90分程度で切断）内に1回で終わらない可能性が高いです。**途中で切れるのは前提**として、何度かに分けて④のセルを再実行する想定で使ってください。
- （任意）⑤のセルは「無操作による切断」を遅らせる非公式の小技です。使わなくても④の再実行だけで完走できますが、放置しておきたい場合に使ってください。

## 使い方
上から順にセルを実行してください（Shift+Enter）。①は初回のみGoogleアカウントの認可が必要です。

## ① Googleドライブをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ② リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ③ ダウンロード先（ドライブ）を決めて、対象ヶ月数を確認

In [ ]:
OUT_DIR = '/content/drive/MyDrive/jra3q_pressure'

!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --dry-run --out-dir "{OUT_DIR}"

## ④ ダウンロード実行（本体）

**途中でセッションが切れたら、①からもう一度実行し直してください。**既にドライブに保存済みのファイルは自動でスキップされ、続きからダウンロードされます。1回で終わらなくて正常です。

地上気圧も欲しい場合は、下のセルの末尾に `--include-surface-pressure` を追加してください。

In [ ]:
!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --out-dir "{OUT_DIR}"

## ⑤ （任意）無操作切断を遅らせる

Colabは無操作が続くと自動切断されることがあります。長時間かかる場合、このセルを実行しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信はせず、④が途中で止まっていたら②→④の順で再実行してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## ⑥ （任意）完了確認

ドライブに保存されたファイル数・合計サイズを確認します。185ファイル（`--include-surface-pressure` を付けた場合は370ファイル）になれば完了です。

In [ ]:
import pathlib

files = list(pathlib.Path(OUT_DIR).glob('*.nc'))
total_gb = sum(f.stat().st_size for f in files) / 1e9
print(f'{len(files)} files, {total_gb:.2f} GB')